# Testing for using FastF1 library for historical F1 data retrieval.

Imports & disable logging/warnings:

In [1]:
import fastf1 as ff1
import numpy as np
import pandas as pd
import os
import warnings
import logging
import time

logging.getLogger("fastf1").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", category=FutureWarning)

Get all qualifying data from 2018-2025:

In [ ]:
cache_dir = "../data/raw/historical/"
os.makedirs(cache_dir, exist_ok=True)
ff1.Cache.enable_cache(cache_dir)

YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
SESSION = "Q"

all_data = []
for year in YEARS:
    schedule = None
    try:
        schedule = ff1.get_event_schedule(year, include_testing=False)
    except Exception as e:
        print(f"Could not load schedule for {year}. Reason: {e}")
        continue

    for _, event in schedule.iterrows():
        try:
            quali = ff1.get_session(year, event.EventName, SESSION)
            quali.load(laps=True)
        except Exception as e:
            print(f"Skipping {year} {event.EventName}. Reason: {e}")
            continue
        
        if quali.laps.empty:
            print(f"No qualifying laps for {year} {event.EventName}. Skipping.")
            continue

        # Store both laps and session info for circuit identification
        all_data.append({
            'laps': quali.laps,
            'event': event.EventName,
            'year': year,
            'location': event.Location if 'Location' in event.index else event.EventName
        })
        
        print(f"Loaded {year} {event.EventName}.")

Loaded 2018 Australian Grand Prix.
Loaded 2018 Bahrain Grand Prix.
Loaded 2018 Chinese Grand Prix.
Loaded 2018 Azerbaijan Grand Prix.
Loaded 2018 Spanish Grand Prix.
Loaded 2018 Monaco Grand Prix.
Loaded 2018 Canadian Grand Prix.
Loaded 2018 French Grand Prix.
Loaded 2018 Austrian Grand Prix.
Loaded 2018 British Grand Prix.
Loaded 2018 German Grand Prix.
Loaded 2018 Hungarian Grand Prix.
Loaded 2018 Belgian Grand Prix.
Loaded 2018 Italian Grand Prix.
Loaded 2018 Singapore Grand Prix.
Loaded 2018 Russian Grand Prix.
Loaded 2018 Japanese Grand Prix.
Loaded 2018 United States Grand Prix.
Loaded 2018 Mexican Grand Prix.
Loaded 2018 Brazilian Grand Prix.
Loaded 2018 Abu Dhabi Grand Prix.
Loaded 2019 Australian Grand Prix.
Loaded 2019 Bahrain Grand Prix.
Loaded 2019 Chinese Grand Prix.
Loaded 2019 Azerbaijan Grand Prix.
Loaded 2019 Spanish Grand Prix.
Loaded 2019 Monaco Grand Prix.
Loaded 2019 Canadian Grand Prix.
Loaded 2019 French Grand Prix.
Loaded 2019 Austrian Grand Prix.
Loaded 2019 Br

Corner detection using multi-signal fusion:

In [3]:
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d

def detect_corners(telemetry, target_corners=None):
    """
    Track-agnostic corner detection using multi-signal fusion.
    Combines lateral acceleration, brake, throttle, speed, and heading signals.
    Uses adaptive thresholding to match target corner count if provided.
    """
    if telemetry.empty or len(telemetry) < 20:
        return []
    
    target_corners = target_corners or 15
    
    # Calculate time delta for derivatives
    time_delta = telemetry['Time'].diff().dt.total_seconds().fillna(0)
    time_delta = time_delta.replace(0, np.nan).fillna(time_delta.median())
    
    # Multi-signal cornering intensity calculation
    # 1. Lateral acceleration (primary signal)
    vx, vy = telemetry['X'].diff() / time_delta, telemetry['Y'].diff() / time_delta
    lateral_accel = np.sqrt((vx.diff() / time_delta)**2 + (vy.diff() / time_delta)**2).fillna(0)
    lateral_norm = uniform_filter1d(lateral_accel, size=7) / (np.max(lateral_accel) + 1e-6)
    
    # 2-5. Supporting signals
    brake = telemetry['Brake'].values.astype(float)
    throttle_reduction = np.maximum(-np.diff(telemetry['Throttle'].values, prepend=telemetry['Throttle'].values[0]), 0) / 100.0
    speed_reduction = np.maximum(-np.diff(telemetry['Speed'].values, prepend=telemetry['Speed'].values[0]), 0)
    speed_norm = speed_reduction / (np.max(speed_reduction) + 1e-6)
    heading_rate = np.abs(np.diff(np.unwrap(np.arctan2(telemetry['Y'].diff(), telemetry['X'].diff()).fillna(0)), prepend=0))
    heading_norm = uniform_filter1d(heading_rate, size=7) / (np.max(heading_rate) + 1e-6)
    
    # Composite cornering intensity score
    cornering_score = uniform_filter1d(
        lateral_norm * 4.0 + brake * 2.0 + throttle_reduction * 1.5 + speed_norm * 2.0 + heading_norm * 1.5,
        size=9
    )
    
    # Adaptive prominence search to hit target corner count
    min_prom, max_prom, best_result = 0.01, 5.0, None
    
    for _ in range(40):
        mid_prom = (min_prom + max_prom) / 2.0
        peaks, _ = find_peaks(cornering_score, prominence=mid_prom, distance=18, width=2)
        
        if best_result is None or abs(len(peaks) - target_corners) < abs(best_result['count'] - target_corners):
            best_result = {'peaks': peaks, 'count': len(peaks), 'prominence': mid_prom}
        
        if len(peaks) == target_corners:
            return peaks.tolist()
        
        if len(peaks) > target_corners:
            min_prom = mid_prom
        else:
            max_prom = mid_prom
        
        if max_prom - min_prom < 1e-5:
            break
    
    # Multi-parameter search if not exact
    if best_result['count'] != target_corners:
        for dist in range(10, 28, 2):
            for width in [1, 2, 3, 4]:
                for prom_mult in np.linspace(0.3, 3.0, 30):
                    peaks, _ = find_peaks(cornering_score, prominence=best_result['prominence'] * prom_mult, distance=dist, width=width)
                    if len(peaks) == target_corners:
                        return peaks.tolist()
    
    # Fallback: select top-N strongest peaks
    if best_result['count'] != target_corners:
        all_peaks, properties = find_peaks(cornering_score, prominence=0.02, distance=15, width=1)
        if len(all_peaks) >= target_corners:
            combined_score = properties['prominences'] * cornering_score[all_peaks]
            top_peaks = all_peaks[np.sort(np.argsort(combined_score)[-target_corners:])]
            return top_peaks.tolist()
    
    return best_result['peaks'].tolist()

Apply corner detection to all circuits:

In [ ]:
# Actual corner counts for F1 circuits
ACTUAL_CORNERS = {
    'Melbourne': 16, 'Sakhir': 15, 'Shanghai': 16, 'Baku': 20, 'Barcelona': 16,
    'Monte Carlo': 19, 'Monaco': 19, 'Montréal': 14, 'Le Castellet': 15,
    'Spielberg': 10, 'Silverstone': 18, 'Hockenheim': 17, 'Budapest': 14,
    'Spa-Francorchamps': 19, 'Monza': 11, 'Singapore': 23, 'Marina Bay': 23,
    'Sochi': 18, 'Suzuka': 18, 'Austin': 20, 'Mexico City': 17, 'São Paulo': 15,
    'Yas Marina': 21, 'Yas Island': 21, 'Mugello': 15, 'Nürburgring': 15,
    'Portimão': 15, 'Imola': 19, 'Istanbul': 14, 'Zandvoort': 14, 'Lusail': 16,
    'Jeddah': 27, 'Miami': 19, 'Miami Gardens': 19, 'Las Vegas': 17
}

# Apply corner detection to all circuits
print(f"{'Circuit':<25} {'Detected':>10} {'Actual':>10} {'Difference':>10}")
print("=" * 60)

segmented_circuits = {}
for session_data in all_data:
    location = session_data['location']
    
    if location in segmented_circuits:
        continue
    
    target = ACTUAL_CORNERS.get(location)
    if not target:
        continue
    
    laps_df = session_data['laps']
    valid_laps = laps_df[laps_df['LapTime'].notna()]
    
    if valid_laps.empty:
        continue
    
    try:
        fastest_lap = valid_laps.loc[valid_laps['LapTime'].idxmin()]
        tel = fastest_lap.get_telemetry()
        corners = detect_corners(tel, target)
        diff = len(corners) - target
        
        segmented_circuits[location] = {
            'Location': location,
            'Event': session_data['event'],
            'Year': session_data['year'],
            'Driver': fastest_lap['Driver'],
            'LapNumber': fastest_lap['LapNumber'],
            'LapTime': fastest_lap['LapTime'],
            'CornersDetected': len(corners),
            'CornerIndices': corners
        }
        
        status = "✓" if diff == 0 else "❌"
        print(f"{location:<25} {len(corners):>10} {target:>10} {diff:>+10} {status}")
    except Exception as e:
        print(f"{location:<25} ERROR: {str(e)[:40]}")

print("=" * 60)
perfect = sum(1 for loc, data in segmented_circuits.items() if ACTUAL_CORNERS.get(loc) == data['CornersDetected'])
total = len(segmented_circuits)
print(f"\nPerfect matches: {perfect}/{total} ({perfect/total*100:.1f}%)")


Circuit                     Detected     Actual Difference
Melbourne                         16         16         +0 ✓
Sakhir                            15         15         +0 ✓
Shanghai                          16         16         +0 ✓
Baku                              20         20         +0 ✓
Barcelona                         16         16         +0 ✓
Monte Carlo                       19         19         +0 ✓
Montréal                          14         14         +0 ✓
Le Castellet                      15         15         +0 ✓
Spielberg                         10         10         +0 ✓
Silverstone                       18         18         +0 ✓
Hockenheim                        14         17         -3 ❌
Budapest                          14         14         +0 ✓
Spa-Francorchamps                 19         19         +0 ✓
Monza                             11         11         +0 ✓
Singapore                         23         23         +0 ✓
Sochi                     